# Pet mobility check (dogs & cats) – gait asymmetry from a phone video

**What it does:** film a dog walking or trotting past the camera, side-on. The notebook finds 39 body points per frame (DeepLabCut **SuperAnimal-Quadruped**, pretrained, no training needed), then measures how evenly the **head** and **pelvis** move up and down with each step.

**Why head and pelvis:** a dog with a sore leg drops its body *less* when that leg takes weight, so the two low points in each stride become unequal (Gómez Álvarez et al. 2017, *Vet J*; Bergh et al. 2018, *Acta Vet Scand*). Head = front legs, pelvis = hind legs.

**Output:** 🟢 looks even · 🟠 keep an eye · 🔴 ask your vet, plus which leg is suspected.
⚠️ **Screening/warning only, not a diagnosis.** Thresholds are placeholders until calibrated on real videos (section 6).

**How to film:** phone held still (or on a tripod) at the dog's height, 3–5 m away, dog walks or trots **across** the frame for at least 3–4 full strides, flat ground, whole dog visible.

**Before running:** `Runtime → Change runtime type → GPU (T4)`.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU - switch runtime to GPU'

In [ ]:
# Takes a few minutes. If Colab asks to restart the session afterwards, restart and continue from the next cell.
!pip install -q --pre deeplabcut

## 2. Analysis code
These cells write the helper modules (same files as `ml/gait/` in the repo).

In [ ]:
%%writefile gait_analysis.py
"""Dog gait asymmetry analysis (mobility check).

Input : per-frame 2D keypoints of one dog filmed side-on while walking/trotting.
Output: head and pelvis vertical asymmetry indices, suspected limb, traffic light.

Method (screening only, NOT a diagnosis):
- Vets read lameness from vertical head and pelvis movement. In each stride the
  head (front limbs) / pelvis (hind limbs) moves down and up twice, once per step.
  With a lame limb the body drops LESS while the painful limb bears weight, so the
  two lowest points in a stride become unequal ("MinDiff").
  Refs: Gomez Alvarez et al. 2017 Vet J 229; Bergh et al. 2018 Acta Vet Scand 60.
- We detect the low points (troughs) of each signal, compare consecutive troughs,
  and express the consistent difference as a % of the up/down amplitude.
- The limb bearing weight at the HIGHER trough is the suspected limb.

Thresholds are placeholders and must be calibrated on real videos.
"""
from __future__ import annotations

import warnings as _warnings
from dataclasses import dataclass, field

import numpy as np
from scipy.signal import find_peaks, savgol_filter

# ---------------------------------------------------------------------------
# Keypoint name mappings. Each role lists candidate names; the first ones found
# in the data are used (several head points are averaged for stability).
# ---------------------------------------------------------------------------
SUPERANIMAL_QUADRUPED = {
    "head": ["left_earbase", "right_earbase", "left_eye", "right_eye"],
    "pelvis": ["tail_base"],
    "withers": ["neck_base", "back_base"],
    "front_left_paw": ["front_left_paw"],
    "front_right_paw": ["front_right_paw"],
    "hind_left_paw": ["back_left_paw"],
    "hind_right_paw": ["back_right_paw"],
}

# Ultralytics Dog-Pose (StanfordExtra scheme) - for a later trained phone model.
DOG_POSE_24 = {
    "head": ["left_ear_base", "right_ear_base", "left_eye", "right_eye"],
    "pelvis": ["tail_start"],
    "withers": ["withers"],
    "front_left_paw": ["front_left_paw"],
    "front_right_paw": ["front_right_paw"],
    "hind_left_paw": ["rear_left_paw"],
    "hind_right_paw": ["rear_right_paw"],
}

# Asymmetry index thresholds in % (PLACEHOLDERS - calibrate on real videos).
GREEN_MAX = 15.0
AMBER_MAX = 30.0
MIN_STEPS = 6            # need at least 3 strides worth of troughs
MIN_LIKELIHOOD = 0.5
MIN_FPS = 20             # below this a step is only a few frames - cannot read it
MIN_AMPLITUDE = 0.01     # up/down movement must be >= 1% of body length to be readable
MIN_CONSISTENCY_T = 2.0  # the high/low alternation must repeat stride after stride (t-statistic)


@dataclass
class SignalResult:
    name: str                      # "head" or "pelvis"
    t: np.ndarray                  # time (s)
    signal: np.ndarray             # detrended vertical position, up = positive, body-length units
    troughs: np.ndarray            # frame indices of low points
    trough_side: list              # "left"/"right"/None: which limb bears weight at each trough
    asymmetry_index: float         # % of amplitude (nan if not enough steps)
    min_diff: float                # consistent trough difference (body-length units)
    amplitude: float               # mean peak-to-trough (body-length units)
    suspected_side: str | None     # side bearing weight at the higher trough
    side_agreement: float          # 0..1 fraction of strides agreeing with the side
    n_steps: int
    consistency_t: float = float("nan")  # how consistently the same step is higher (|t| >= 2 = consistent)
    readable: bool = True                # False if movement too small / too few steps


@dataclass
class GaitResult:
    head: SignalResult
    pelvis: SignalResult
    light: str                     # "green" / "amber" / "red" / "retake"
    suspected_limb: str | None     # e.g. "front left"
    message: str
    warnings: list = field(default_factory=list)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def _pick(kps: dict, names: list[str]):
    """Average the available keypoints from `names`. kps[name] = (x, y, p) arrays."""
    found = [kps[n] for n in names if n in kps]
    if not found:
        return None
    xs = np.stack([f[0] for f in found])
    ys = np.stack([f[1] for f in found])
    ps = np.stack([f[2] for f in found])
    good = ps >= MIN_LIKELIHOOD
    with np.errstate(invalid="ignore"), _warnings.catch_warnings():
        _warnings.simplefilter("ignore", category=RuntimeWarning)
        x = np.where(good, xs, np.nan)
        y = np.where(good, ys, np.nan)
        x = np.nanmean(x, axis=0) if np.any(good) else np.full(xs.shape[1], np.nan)
        y = np.nanmean(y, axis=0) if np.any(good) else np.full(ys.shape[1], np.nan)
    return x, y


def _fill(a: np.ndarray, max_gap: int) -> np.ndarray:
    """Linearly interpolate NaN gaps up to max_gap frames; longer gaps stay NaN."""
    a = a.astype(float).copy()
    n = len(a)
    idx = np.arange(n)
    good = ~np.isnan(a)
    if good.sum() < 2:
        return a
    filled = np.interp(idx, idx[good], a[good])
    # keep long gaps as NaN
    isnan = np.isnan(a)
    start = None
    for i in range(n + 1):
        if i < n and isnan[i]:
            start = i if start is None else start
        elif start is not None:
            if i - start > max_gap or start == 0 or i == n:
                filled[start:i] = np.nan
            start = None
    return filled


def _smooth(a: np.ndarray, fps: float) -> np.ndarray:
    win = max(5, int(round(fps * 0.1)) | 1)  # ~0.1 s, odd
    out = a.copy()
    good = ~np.isnan(a)
    if good.sum() > win:
        out[good] = savgol_filter(a[good], win, 2)
    return out


def _detrend(a: np.ndarray, fps: float, window_s: float) -> np.ndarray:
    """Remove slow drift (camera shake, ground slope) with a moving average."""
    w = max(3, int(round(fps * window_s)))
    good = ~np.isnan(a)
    filled = np.where(good, a, np.nanmean(a))
    kernel = np.ones(w) / w
    trend = np.convolve(np.pad(filled, (w // 2, w - 1 - w // 2), mode="edge"), kernel, mode="valid")
    out = a - trend
    out[~good] = np.nan
    return out


def _stance_side(kps_paws, frame: int, fps: float) -> str | None:
    """Which paw (left/right) is on the ground at `frame`: the one moving slowest horizontally."""
    (lx, _), (rx, _) = kps_paws
    h = max(1, int(round(fps * 0.03)))
    lo, hi = max(0, frame - h), min(len(lx) - 1, frame + h)
    if hi <= lo:
        return None
    vl = abs(lx[hi] - lx[lo])
    vr = abs(rx[hi] - rx[lo])
    if np.isnan(vl) or np.isnan(vr) or abs(vl - vr) < 1e-9:
        return None
    return "left" if vl < vr else "right"


def _analyse_signal(name, y_up, fps, paws, step_period_s) -> SignalResult:
    t = np.arange(len(y_up)) / fps
    sig = _detrend(y_up, fps, window_s=2 * step_period_s)  # one stride = 2 steps
    clean = np.where(np.isnan(sig), 0.0, sig)

    min_dist = max(2, int(round(fps * step_period_s * 0.6)))
    prom = 0.15 * np.nanstd(sig) if np.nanstd(sig) > 0 else None
    troughs, _ = find_peaks(-clean, distance=min_dist, prominence=prom)
    peaks, _ = find_peaks(clean, distance=min_dist, prominence=prom)
    troughs = np.array([i for i in troughs if not np.isnan(sig[i])], dtype=int)

    n = len(troughs)
    res = SignalResult(name, t, sig, troughs, [], float("nan"), float("nan"),
                       float("nan"), None, 0.0, n)
    if n < MIN_STEPS:
        res.readable = False
        return res

    vals = sig[troughs]
    # amplitude: mean high point minus mean low point (independent of which side is lame)
    pk = [sig[p] for p in peaks if not np.isnan(sig[p])]
    amplitude = float(np.mean(pk) - np.mean(vals)) if pk else float(np.nanstd(sig) * 2)

    # consistent alternating difference: even-indexed vs odd-indexed troughs
    pairs = [(vals[i], vals[i + 1]) for i in range(0, n - 1, 2)]
    d = np.array([a - b for a, b in pairs])          # even minus odd
    min_diff = float(np.mean(d))
    asym = abs(min_diff) / amplitude * 100 if amplitude > 0 else float("nan")
    if len(d) >= 2 and np.std(d, ddof=1) > 0:
        t_stat = float(np.mean(d) / (np.std(d, ddof=1) / np.sqrt(len(d))))
    else:
        t_stat = float("inf") if abs(min_diff) > 0 else 0.0
    res.consistency_t = t_stat
    res.readable = amplitude >= MIN_AMPLITUDE

    sides = [_stance_side(paws, int(i), fps) if paws else None for i in troughs]
    # higher trough = less drop = suspected painful limb in stance
    hi_parity = 0 if min_diff > 0 else 1
    hi_sides = [s for k, s in enumerate(sides) if k % 2 == hi_parity and s]
    if hi_sides:
        left = hi_sides.count("left")
        side = "left" if left >= len(hi_sides) - left else "right"
        agree = max(left, len(hi_sides) - left) / len(hi_sides)
    else:
        side, agree = None, 0.0

    res.trough_side = sides
    res.asymmetry_index = float(asym)
    res.min_diff = min_diff
    res.amplitude = amplitude
    res.suspected_side = side
    res.side_agreement = float(agree)
    return res


def _estimate_step_period(y_up: np.ndarray, fps: float) -> float:
    """Dominant up/down period from autocorrelation (dogs: roughly 0.15-0.6 s per step)."""
    a = y_up[~np.isnan(y_up)]
    if len(a) < fps:
        return 0.3
    a = a - a.mean()
    ac = np.correlate(a, a, mode="full")[len(a) - 1:]
    lo, hi = int(fps * 0.12), int(fps * 0.8)
    if hi <= lo + 1 or hi >= len(ac):
        return 0.3
    seg = ac[lo:hi]
    # first clear local maximum = one step (the stride peak comes later at ~2x)
    for i in range(1, len(seg) - 1):
        if seg[i] >= seg[i - 1] and seg[i] >= seg[i + 1] and seg[i] > 0.2 * ac[0]:
            return (lo + i) / fps
    return (lo + int(np.argmax(seg))) / fps


def _signal_light(s: SignalResult) -> str:
    if not s.readable or np.isnan(s.asymmetry_index):
        return "retake"
    if abs(s.consistency_t) < MIN_CONSISTENCY_T:
        return "green"   # no consistent high/low pattern -> nothing to flag
    return _light(s.asymmetry_index)


def _light(index: float) -> str:
    if np.isnan(index):
        return "retake"
    if index <= GREEN_MAX:
        return "green"
    if index <= AMBER_MAX:
        return "amber"
    return "red"


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------
def analyse_gait(kps: dict, fps: float, mapping: dict = SUPERANIMAL_QUADRUPED) -> GaitResult:
    """kps: {bodypart_name: (x[n], y[n], likelihood[n])} in pixels, image y pointing down."""
    warnings = []
    if fps < MIN_FPS:
        empty = SignalResult("head", np.array([]), np.array([]), np.array([], int), [], float("nan"),
                             float("nan"), float("nan"), None, 0.0, 0, readable=False)
        p_empty = SignalResult("pelvis", np.array([]), np.array([]), np.array([], int), [], float("nan"),
                               float("nan"), float("nan"), None, 0.0, 0, readable=False)
        return GaitResult(empty, p_empty, "retake", None,
                          f"Video frame rate is too low ({fps:.0f} fps). Please film at 30 fps or more.",
                          [f"Frame rate {fps:.0f} fps < {MIN_FPS} fps."])
    max_gap = int(round(fps * 0.2))

    def role(r):
        p = _pick(kps, mapping[r])
        if p is None:
            return None
        return _smooth(_fill(p[0], max_gap), fps), _smooth(_fill(p[1], max_gap), fps)

    head, pelvis, withers = role("head"), role("pelvis"), role("withers")
    if head is None or pelvis is None:
        raise ValueError("Head or pelvis keypoints missing - check the keypoint mapping.")

    # body length for scale (pixels) -> signals in body-length units
    ref = withers if withers is not None else head
    body_len = np.nanmedian(np.hypot(ref[0] - pelvis[0], ref[1] - pelvis[1]))
    if not np.isfinite(body_len) or body_len <= 0:
        body_len = 1.0
        warnings.append("Could not estimate body length; results not scale-normalised.")

    head_up = -head[1] / body_len
    pelvis_up = -pelvis[1] / body_len

    # quality: dog should cross the frame side-on
    centre_x = (head[0] + pelvis[0]) / 2
    travel = (np.nanmax(centre_x) - np.nanmin(centre_x)) / body_len
    if travel < 1.5:
        warnings.append("Dog moved less than 1.5 body lengths across the frame - film side-on, walking past the camera.")
    coverage = np.mean(~np.isnan(head_up) & ~np.isnan(pelvis_up))
    if coverage < 0.6:
        warnings.append(f"Dog tracked in only {coverage:.0%} of frames.")

    step = _estimate_step_period(pelvis_up, fps)

    def paws(l, r):
        a, b = role(l), role(r)
        return (a, b) if a is not None and b is not None else None

    h = _analyse_signal("head", head_up, fps, paws("front_left_paw", "front_right_paw"), step)
    p = _analyse_signal("pelvis", pelvis_up, fps, paws("hind_left_paw", "hind_right_paw"), step)

    if h.n_steps < MIN_STEPS and p.n_steps < MIN_STEPS:
        return GaitResult(h, p, "retake", None,
                          "Not enough steps detected. Film at least 3 full strides side-on.",
                          warnings)
    if not h.readable and not p.readable:
        warnings.append(f"Up/down movement too small to read (head {h.amplitude*100:.1f}%, "
                        f"pelvis {p.amplitude*100:.1f}% of body length).")
        return GaitResult(h, p, "retake", None,
                          "Could not see clear up/down movement. Film closer, side-on, with the camera still.",
                          warnings)

    lh, lp = _signal_light(h), _signal_light(p)
    order = {"retake": 0, "green": 1, "amber": 2, "red": 3}
    light = max([lh, lp], key=lambda s: order[s])

    limb = None
    if light in ("amber", "red"):
        worst = h if order[lh] >= order[lp] and (np.nan_to_num(h.asymmetry_index) >= np.nan_to_num(p.asymmetry_index)) else p
        end = "front" if worst.name == "head" else "hind"
        limb = f"{end} {worst.suspected_side}" if worst.suspected_side else end
        if worst.side_agreement < 0.7:
            warnings.append("Side (left/right) is uncertain - paws were hard to tell apart.")
        if worst.name == "head" and order[lp] >= 2:
            warnings.append("Head nod can also be compensation for a hind-limb problem.")

    messages = {
        "green": "Movement looks even. Keep checking regularly.",
        "amber": f"Slightly uneven movement{' (' + limb + ')' if limb else ''}. Re-check in a few days; book your vet if it persists.",
        "red": f"Clearly uneven movement{' (' + limb + ')' if limb else ''}. We recommend a check with your vet.",
        "retake": "Could not read the movement reliably. Please film again side-on.",
    }
    return GaitResult(h, p, light, limb, messages[light], warnings)


def keypoints_from_dlc_h5(h5_path: str, individual: str | None = None):
    """Load a DeepLabCut output .h5 into {bodypart: (x, y, likelihood)} for one dog.

    Picks the individual with the highest mean likelihood when several are present.
    """
    import pandas as pd

    df = pd.read_hdf(h5_path)
    cols = df.columns
    names = list(cols.names)
    if "individuals" in names:
        inds = cols.get_level_values("individuals").unique()
        if individual is None:
            best, best_p = None, -1.0
            for ind in inds:
                sub = df.xs(ind, axis=1, level="individuals")
                p = sub.xs("likelihood", axis=1, level="coords").mean().mean()
                if p > best_p:
                    best, best_p = ind, p
            individual = best
        df = df.xs(individual, axis=1, level="individuals")
    out = {}
    for bp in df.columns.get_level_values("bodyparts").unique():
        sub = df.xs(bp, axis=1, level="bodyparts")
        g = lambda c: sub.xs(c, axis=1, level="coords").iloc[:, 0].to_numpy(dtype=float)
        out[bp] = (g("x"), g("y"), g("likelihood"))
    return out


In [ ]:
%%writefile visualize.py
"""Plots and annotated result video for the mobility check."""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

LIGHT_BGR = {"green": (80, 175, 76), "amber": (0, 165, 255), "red": (60, 60, 230), "retake": (150, 150, 150)}
LIGHT_HEX = {"green": "#4caf50", "amber": "#ffa500", "red": "#e63c3c", "retake": "#969696"}

# SuperAnimal-Quadruped skeleton (subset that is useful from a side view)
SKELETON_SA = [
    ("nose", "left_eye"), ("left_eye", "left_earbase"), ("left_earbase", "neck_base"),
    ("neck_base", "back_middle"), ("back_middle", "tail_base"), ("tail_base", "tail_end"),
    ("neck_base", "front_left_thai"), ("front_left_thai", "front_left_knee"), ("front_left_knee", "front_left_paw"),
    ("neck_base", "front_right_thai"), ("front_right_thai", "front_right_knee"), ("front_right_knee", "front_right_paw"),
    ("tail_base", "back_left_thai"), ("back_left_thai", "back_left_knee"), ("back_left_knee", "back_left_paw"),
    ("tail_base", "back_right_thai"), ("back_right_thai", "back_right_knee"), ("back_right_knee", "back_right_paw"),
]
LEFT_COLOR, RIGHT_COLOR = (255, 170, 0), (255, 0, 200)   # BGR: left = blue-ish, right = magenta


def plot_signals(result, title: str = "Mobility check"):
    """Head and pelvis up/down curves with the low points coloured by weight-bearing side."""
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    for ax, s, label in [(axes[0], result.head, "Head (front legs)"), (axes[1], result.pelvis, "Pelvis (hind legs)")]:
        ax.plot(s.t, s.signal * 100, color="#555", lw=1.5)
        for k, i in enumerate(s.troughs):
            side = s.trough_side[k] if k < len(s.trough_side) else None
            c = "#1f77b4" if side == "left" else "#d62728" if side == "right" else "#999"
            ax.plot(s.t[i], s.signal[i] * 100, "o", color=c, ms=7)
        idx = "n/a" if np.isnan(s.asymmetry_index) else f"{s.asymmetry_index:.0f}%"
        ax.set_title(f"{label}: asymmetry {idx}  (steps: {s.n_steps})", loc="left", fontsize=11)
        ax.set_ylabel("up / down\n(% body length)")
        ax.grid(alpha=0.3)
    axes[1].set_xlabel("time (s)")
    axes[0].plot([], [], "o", color="#1f77b4", label="left leg on ground")
    axes[0].plot([], [], "o", color="#d62728", label="right leg on ground")
    axes[0].legend(loc="upper right", fontsize=9)
    fig.suptitle(f"{title}: {result.light.upper()}  {result.suspected_limb or ''}",
                 color=LIGHT_HEX[result.light], fontsize=14, fontweight="bold")
    fig.tight_layout()
    return fig


@dataclass
class DemoLabel:
    """Hand-written banner for pitch videos (DEMO MODE - not the model's own verdict)."""
    light: str = "red"
    title: str = "Ask your vet"
    detail: str = "Uneven movement detected"


def render_video(video_path: str, kps: dict, result, out_path: str,
                 skeleton=SKELETON_SA, min_p: float = 0.5, banner: bool = True,
                 demo_label: "DemoLabel | None" = None):
    """Write a copy of the video with skeleton and (optionally) a result banner.

    `result` can be a dog GaitResult, a CatResult, or None (skeleton only).
    `demo_label` replaces the banner with a hand-written one (shows a small 'DEMO' tag)."""
    import cv2

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    show_banner = banner and (result is not None or demo_label is not None)
    colour = LIGHT_BGR[(demo_label or result).light] if show_banner else None
    scale = max(0.6, min(w, h) / 720)
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        _draw_frame(frame, kps, i, skeleton, min_p, scale)
        if show_banner and demo_label is not None:
            _draw_demo_banner(frame, demo_label, colour, scale)
        elif show_banner:
            _draw_banner(frame, result, colour, scale)
        writer.write(frame)
        i += 1
    cap.release()
    writer.release()
    return out_path


def _pt(kps, name, i, min_p):
    if name not in kps or i >= len(kps[name][0]):
        return None
    x, y, p = kps[name][0][i], kps[name][1][i], kps[name][2][i]
    if np.isnan(x) or np.isnan(y) or p < min_p:
        return None
    return int(x), int(y)


def _draw_frame(frame, kps, i, skeleton, min_p, scale):
    import cv2

    th = max(2, int(3 * scale))
    for a, b in skeleton:
        pa, pb = _pt(kps, a, i, min_p), _pt(kps, b, i, min_p)
        if pa and pb:
            c = LEFT_COLOR if "left" in b else RIGHT_COLOR if "right" in b else (230, 230, 230)
            cv2.line(frame, pa, pb, c, th, cv2.LINE_AA)
    for name in kps:
        p = _pt(kps, name, i, min_p)
        if p:
            cv2.circle(frame, p, th + 1, (255, 255, 255), -1, cv2.LINE_AA)
    # highlight the two measured points
    for name, c in [("left_earbase", (0, 255, 255)), ("tail_base", (0, 255, 255))]:
        p = _pt(kps, name, i, min_p)
        if p:
            cv2.circle(frame, p, th * 3, c, th, cv2.LINE_AA)


def _draw_demo_banner(frame, d: DemoLabel, colour, scale):
    import cv2

    h, w = frame.shape[:2]
    bh = int(70 * scale)
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, bh), (30, 30, 30), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    cv2.circle(frame, (int(35 * scale), bh // 2), int(20 * scale), colour, -1, cv2.LINE_AA)
    cv2.putText(frame, d.title, (int(70 * scale), int(bh * 0.45)), cv2.FONT_HERSHEY_SIMPLEX, 0.9 * scale,
                (255, 255, 255), max(1, int(2 * scale)), cv2.LINE_AA)
    cv2.putText(frame, f"{d.detail}   Screening only - not a diagnosis", (int(70 * scale), int(bh * 0.85)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5 * scale, (220, 220, 220), 1, cv2.LINE_AA)
    cv2.putText(frame, "DEMO", (w - int(70 * scale), h - int(12 * scale)), cv2.FONT_HERSHEY_SIMPLEX,
                0.5 * scale, (200, 200, 200), 1, cv2.LINE_AA)


def _draw_banner(frame, result, colour, scale):
    import cv2

    h, w = frame.shape[:2]
    bh = int(70 * scale)
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, bh), (30, 30, 30), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    cv2.circle(frame, (int(35 * scale), bh // 2), int(20 * scale), colour, -1, cv2.LINE_AA)
    label = {"green": "Looks even", "amber": "Keep an eye", "red": "Ask your vet", "retake": "Please re-film"}[result.light]
    txt = label + (f"  |  {result.suspected_limb}" if result.suspected_limb else "")
    cv2.putText(frame, txt, (int(70 * scale), int(bh * 0.45)), cv2.FONT_HERSHEY_SIMPLEX, 0.9 * scale,
                (255, 255, 255), max(1, int(2 * scale)), cv2.LINE_AA)
    fmt = lambda v: "n/a" if np.isnan(v) else f"{v:.0f}%"
    gait = result if hasattr(result, "head") else getattr(result, "gait", None)
    if gait is not None and getattr(result, "mode", "walk") == "walk":
        sub = (f"Head asym {fmt(gait.head.asymmetry_index)}   Pelvis asym {fmt(gait.pelvis.asymmetry_index)}"
               "   Screening only - not a diagnosis")
    else:
        msg = result.message.split(". ")[0]
        sub = f"{msg[:70]}   Screening only - not a diagnosis"
    cv2.putText(frame, sub, (int(70 * scale), int(bh * 0.85)), cv2.FONT_HERSHEY_SIMPLEX, 0.5 * scale,
                (220, 220, 220), 1, cv2.LINE_AA)


In [ ]:
%%writefile synthetic.py
"""Synthetic trotting-dog keypoints with a controllable lame limb (for testing only)."""
import numpy as np


def make_trot(lame: str | None = None, severity: float = 0.5, seconds: float = 6.0,
              fps: float = 30.0, step: float = 0.25, body_px: float = 200.0,
              speed_bl: float = 1.5, noise_px: float = 1.0, drift_px: float = 10.0,
              direction: int = 1, seed: int = 0):
    """Return ({bodypart: (x, y, p)}, fps) using SuperAnimal names.

    Trot: diagonal pairs LF+RH, then RF+LH, each in stance for one `step`.
    Head trough happens mid-stance of each front limb; pelvis trough mid-stance of
    each hind limb. A lame limb makes the trough during its stance shallower by
    `severity` (0 = sound, 1 = no drop at all).
    """
    rng = np.random.default_rng(seed)
    n = int(seconds * fps)
    t = np.arange(n) / fps
    v = speed_bl * body_px * direction                 # px/s
    cx = 100 + v * t if direction > 0 else 1800 + v * t
    drift = drift_px * np.sin(2 * np.pi * t / 3.0)      # slow camera bob

    phase = (t % (2 * step)) / step                    # 0..2 within a stride
    lf_stance = phase < 1                              # LF + RH stance
    amp_head, amp_pelvis = 0.06 * body_px, 0.04 * body_px

    def bob(amp, stance_a, lame_a: bool, lame_b: bool):
        # one down-up per step: trough at mid-stance
        base = -amp * np.cos(2 * np.pi * (t - step / 2) / step)   # up-positive
        scale = np.ones(n)
        down = base < 0
        if lame_a:
            scale[down & stance_a] = 1 - severity
        if lame_b:
            scale[down & ~stance_a] = 1 - severity
        return base * scale

    head_up = bob(amp_head, lf_stance, lame == "front_left", lame == "front_right")
    # hind: LH stance when RF stance (i.e. ~lf_stance)
    pelvis_up = bob(amp_pelvis, ~lf_stance, lame == "hind_left", lame == "hind_right")

    def paw_x(stance_mask, offset):
        vel = np.where(stance_mask, 0.0, 2 * v)
        return cx[0] + offset + np.cumsum(vel) / fps

    out = {}
    def add(name, x, y):
        out[name] = (x + rng.normal(0, noise_px, n),
                     y + drift + rng.normal(0, noise_px, n),
                     np.full(n, 0.9))

    front = 0.45 * body_px * direction
    back = -0.45 * body_px * direction
    ground, back_y = 600.0, 600.0 - 0.6 * body_px
    head_y = back_y - 0.25 * body_px - head_up
    for nm, dx in [("left_earbase", 0), ("right_earbase", 3), ("left_eye", 15 * direction), ("right_eye", 17 * direction)]:
        add(nm, cx + front + 0.15 * body_px * direction + dx, head_y + (8 if "eye" in nm else 0))
    add("neck_base", cx + front, back_y - pelvis_up * 0 - head_up * 0.3)
    add("tail_base", cx + back, back_y - pelvis_up)
    add("front_left_paw", paw_x(lf_stance, front), np.full(n, ground))
    add("front_right_paw", paw_x(~lf_stance, front + 5), np.full(n, ground))
    add("back_left_paw", paw_x(~lf_stance, back), np.full(n, ground))
    add("back_right_paw", paw_x(lf_stance, back + 5), np.full(n, ground))
    return out, fps


### Self-test (no video, no GPU needed)
A simulated dog that is lame on the front-left leg. Expect 🔴 *front left*.

In [ ]:
%matplotlib inline
import importlib, gait_analysis, visualize, synthetic
for m in (gait_analysis, visualize, synthetic): importlib.reload(m)
from gait_analysis import analyse_gait, keypoints_from_dlc_h5
from visualize import plot_signals, render_video
from synthetic import make_trot

for lame in [None, "front_left", "hind_right"]:
    r = analyse_gait(*make_trot(lame=lame, severity=0.8))
    print(f"simulated: {str(lame):11s} -> {r.light:6s} {r.suspected_limb}  "
          f"(head {r.head.asymmetry_index:.0f}%, pelvis {r.pelvis.asymmetry_index:.0f}%)")
plot_signals(analyse_gait(*make_trot(lame="front_left", severity=0.8)), "Simulated dog");

## 3. Get a video
Either paste a direct video URL, or leave it empty to upload a file (mp4/mov) from your computer.

Open-licence test clips (supplementary videos of PLOS ONE 2013, doi:10.1371/journal.pone.0059875, CC BY 2.5, via Wikimedia Commons):
- healthy dog, side view: `https://upload.wikimedia.org/wikipedia/commons/9/95/Quantitative-Phenotyping-of-Duchenne-Muscular-Dystrophy-Dogs-by-Comprehensive-Gait-Analysis-and-pone.0059875.s003.ogv`
- dog with abnormal (dystrophic) gait: `https://upload.wikimedia.org/wikipedia/commons/8/84/Quantitative-Phenotyping-of-Duchenne-Muscular-Dystrophy-Dogs-by-Comprehensive-Gait-Analysis-and-pone.0059875.s002.ogv`

In [ ]:
import os, subprocess
from google.colab import files

video_url = ""  # @param {type:"string"}
if video_url:
    raw_path = "/content/" + os.path.basename(video_url.split("?")[0])
    subprocess.run(["curl", "-sSfL", "-A", "gait-demo-notebook/0.1", "-o", raw_path, video_url], check=True)
    assert os.path.getsize(raw_path) > 0, "download failed (empty file)"
else:
    uploaded = files.upload()
    raw_path = os.path.abspath(next(iter(uploaded)))
print("video:", raw_path, os.path.getsize(raw_path), "bytes")

In [ ]:
# Normalise: max 1280 px wide, H.264, no audio, optional trim. Smaller = much faster pose estimation.
start_s = 0      # @param {type:"number"}
duration_s = 0   # @param {type:"number"}  (0 = whole video)
os.makedirs("/content/work", exist_ok=True)
video_path = "/content/work/" + os.path.splitext(os.path.basename(raw_path))[0].replace(" ", "_") + ".mp4"
trim = (["-ss", str(start_s)] if start_s else []) + (["-t", str(duration_s)] if duration_s else [])
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", *trim, "-i", raw_path,
                "-vf", "scale='min(1280,iw)':-2", "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", video_path], check=True)
print("ready:", video_path)

## 4. Find the dog's body points (SuperAnimal-Quadruped)
`video_adapt=True` reduces jitter (recommended for the final demo) but takes longer.

In [ ]:
import glob
from deeplabcut.modelzoo.video_inference import video_inference_superanimal

video_adapt = False  # @param {type:"boolean"}
out_dir = "/content/work/results"
os.makedirs(out_dir, exist_ok=True)

_ = video_inference_superanimal(
    videos=[video_path],
    superanimal_name="superanimal_quadruped",
    model_name="hrnet_w32",
    detector_name="fasterrcnn_resnet50_fpn_v2",
    video_adapt=video_adapt,
    max_individuals=1,
    dest_folder=out_dir,
)
stem = os.path.splitext(os.path.basename(video_path))[0]
h5s = sorted(glob.glob(f"{out_dir}/{stem}*.h5"), key=os.path.getmtime)
print("\n".join(h5s))
h5_path = h5s[-1]  # newest (after_adapt if adaptation was used)

## 5. Gait analysis

In [ ]:
import cv2
fps = cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FPS)
kps = keypoints_from_dlc_h5(h5_path)
result = analyse_gait(kps, fps)

icon = {"green": "🟢", "amber": "🟠", "red": "🔴", "retake": "⚪"}[result.light]
print(f"{icon} {result.light.upper()}  {result.suspected_limb or ''}")
print(result.message)
print(f"head asymmetry  : {result.head.asymmetry_index:.1f}%  ({result.head.n_steps} steps, side agreement {result.head.side_agreement:.0%})")
print(f"pelvis asymmetry: {result.pelvis.asymmetry_index:.1f}%  ({result.pelvis.n_steps} steps, side agreement {result.pelvis.side_agreement:.0%})")
for w in result.warnings:
    print("⚠", w)
plot_signals(result, stem);

In [ ]:
# Annotated result video (skeleton + banner), converted to H.264 so it plays in the browser
from IPython.display import HTML
from base64 import b64encode

tmp = f"/content/work/{stem}_annot_raw.mp4"
annotated = f"/content/work/{stem}_mobility.mp4"
render_video(video_path, kps, result, tmp)
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp, "-c:v", "libx264", "-pix_fmt", "yuv420p", annotated], check=True)
HTML(f'<video width=720 controls autoplay loop muted src="data:video/mp4;base64,{b64encode(open(annotated, "rb").read()).decode()}"></video>')

In [ ]:
files.download(annotated)

## 6. Cat mode 🐈
Cats rarely limp visibly: most cats with arthritis just shift weight off the sore leg and stop jumping (Stadig et al. 2016, *Acta Vet Scand*; Slingerland et al. 2011, *Vet J*: 61% of cats ≥6 y have OA). So for cats we add two checks:
- **walk**: head/pelvis symmetry (as dogs) **+ stance time** per paw (a sore paw stays down longer).
- **jump**: film the cat jumping onto a sofa/bed → hesitation before take-off, one smooth jump vs two-stage climb, aborted attempts, hind push-off timing.

Use sections 3–4 with a cat video first (SuperAnimal-Quadruped covers cats), then run below. Thresholds are placeholders.

In [ ]:
%%writefile cat_analysis.py
"""Cat mobility checks: walk (head/pelvis + stance time) and jump.

Input : per-frame 2D keypoints of one cat filmed side-on (DeepLabCut
        SuperAnimal-Quadruped names), {bodypart: (x[n], y[n], likelihood[n])},
        pixels, image y pointing down.
Output: traffic light + plain-English owner message. Screening only, NOT a diagnosis.

Why cats need more than the dog check (see research/cat-gait-literature.md):
- Cats with joint disease rarely limp visibly. Instead they shift weight off the
  sore limb: that limb carries LESS weight but stays on the ground LONGER
  (Stadig, Lascelles & Bergh 2016, Acta Vet Scand 58). -> stance-time asymmetry.
- Owners notice that cats stop jumping, hesitate or climb in stages
  (FMPI, Benito et al. 2013; Godfrey 2002; Slingerland et al. 2011). -> jump check.

Checks
1. analyse_stance: a paw is "in stance" while its horizontal speed is near zero
   (below STANCE_SPEED_FRAC x the body's walking speed, both in body lengths/s).
   Mean stance duration per paw; left/right asymmetry index for the front pair and
   the hind pair: |L - R| / mean(L, R) x 100 %. The side with the LONGER stance is
   the suspected limb.
2. analyse_jump: body centre = mean of withers (neck_base/back_base) and tail_base,
   vertical position in body lengths.
   - take-off  = start of the first fast rise (upward speed > RISE_SPEED_BL).
   - landing   = end of the last rise, once the centre settles on a higher plateau.
   - stages    = separate fast rises with a pause of >= PAUSE_MIN_S between them.
   - hesitation = length of the period just before take-off in which the cat is
     still (centre speed < STILL_SPEED_BL) or crouched (centre lowered by more than
     CROUCH_DROP_BL below its standing height). Counted back from take-off until the
     cat was last walking upright, or to the start of the clip.
     Standing height = 90th percentile of centre height before take-off, so the clip
     should start with the cat walking or standing.
   - aborted attempt = a crouch (>= MIN_CROUCH_S) after which the cat stands up
     again without taking off.
   - hind push-off difference = gap between the times the two hind paws lift off.
3. analyse_cat: combines the above into one result.

All thresholds marked PLACEHOLDER must be calibrated on real cat videos.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
from scipy.signal import savgol_filter

from gait_analysis import (SUPERANIMAL_QUADRUPED, GaitResult, _fill, _pick,
                           _smooth, analyse_gait)

ORDER = {"retake": 0, "green": 1, "amber": 2, "red": 3}
PAWS = ("front_left", "front_right", "hind_left", "hind_right")
DISCLAIMER = "This is a warning, not a diagnosis."

# --- stance thresholds (PLACEHOLDERS - calibrate on real videos) -------------
STANCE_SPEED_FRAC = 0.3      # paw slower than 30% of body speed = on the ground
MIN_STANCE_S = 0.08          # shorter "stances" are tracking noise
MAX_STANCE_S = 2.0           # longer = cat stopped, not a walking stance
MIN_STANCES_PER_PAW = 3
MIN_WALK_SPEED_BL = 0.2      # body lengths / s
STANCE_GREEN_MAX = 10.0      # % stance-duration asymmetry
STANCE_AMBER_MAX = 20.0

# --- jump thresholds (PLACEHOLDERS - calibrate on real videos) ---------------
MIN_CLIP_S = 1.5
RISE_SPEED_BL = 1.0          # upward speed counted as "jumping/climbing", BL/s
MIN_STAGE_RISE_BL = 0.2      # a rise smaller than this is not a stage
MIN_JUMP_RISE_BL = 0.5       # total rise needed to call it a jump
PAUSE_MIN_S = 0.15           # pause between rises that makes it a two-stage climb
SETTLE_SPEED_BL = 0.5        # |vertical speed| below this = landed / settled
STILL_SPEED_BL = 0.3         # centre speed below this = standing still
CROUCH_DROP_BL = 0.08        # centre lowered this much below standing = crouched
MIN_CROUCH_S = 0.3
ABORT_GAP_S = 0.5            # crouch ending this long before take-off = aborted
LIFT_HEIGHT_BL = 0.05        # paw this far above its ground level = lifted off
HESITATION_GREEN_MAX_S = 1.5
HESITATION_AMBER_MAX_S = 3.0
PUSH_OFF_DIFF_MAX_S = 0.06


# ---------------------------------------------------------------------------
# Result types
# ---------------------------------------------------------------------------
@dataclass
class StanceResult:
    mean_stance_s: dict            # paw -> mean stance duration (s), nan if unknown
    n_stances: dict                # paw -> number of complete stances used
    stances: dict                  # paw -> list of (start_s, end_s)
    front_asymmetry: float         # % (nan if not enough stances)
    hind_asymmetry: float          # %
    front_hind_ratio: float        # mean front stance / mean hind stance
    suspected_limb: str | None     # e.g. "hind left" (side with the longer stance)
    light: str                     # green / amber / red / retake
    body_speed: float              # body lengths per second
    warnings: list = field(default_factory=list)


@dataclass
class JumpResult:
    light: str                     # green / amber / red / retake
    message: str
    jumped: bool
    takeoff_s: float               # nan if no jump
    landing_s: float
    hesitation_s: float            # still/crouched time before take-off
    jump_height_bl: float          # landing plateau minus standing height, body lengths
    n_stages: int                  # number of separate rises (1 = one continuous jump)
    two_stage: bool
    aborted_attempts: int
    push_off_diff_s: float         # |left - right| hind lift-off time, nan if unknown
    flags: list                    # short owner-facing findings
    t: np.ndarray                  # time (s)
    centre_up: np.ndarray          # body centre height, body lengths (up = positive)
    warnings: list = field(default_factory=list)


@dataclass
class CatResult:
    mode: str                      # "walk" or "jump"
    light: str
    suspected_limb: str | None
    message: str
    gait: GaitResult | None = None
    stance: StanceResult | None = None
    jump: JumpResult | None = None
    warnings: list = field(default_factory=list)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def _role(kps: dict, mapping: dict, role: str, fps: float):
    names = mapping.get(role)
    if not names:
        return None
    p = _pick(kps, names)
    if p is None:
        return None
    max_gap = int(round(fps * 0.2))
    return _smooth(_fill(p[0], max_gap), fps), _smooth(_fill(p[1], max_gap), fps)


def _body_and_scale(kps, fps, mapping, warnings):
    withers = _role(kps, mapping, "withers", fps)
    pelvis = _role(kps, mapping, "pelvis", fps)
    if pelvis is None:
        raise ValueError("Tail base keypoint missing - check the keypoint mapping.")
    if withers is None:
        withers = _role(kps, mapping, "head", fps)
    if withers is None:
        raise ValueError("Withers/head keypoints missing - check the keypoint mapping.")
    body_len = np.nanmedian(np.hypot(withers[0] - pelvis[0], withers[1] - pelvis[1]))
    if not np.isfinite(body_len) or body_len <= 0:
        body_len = 1.0
        warnings.append("Could not estimate body length; results not scale-normalised.")
    cx = (withers[0] + pelvis[0]) / 2
    cy = (withers[1] + pelvis[1]) / 2
    return cx, cy, float(body_len)


def _deriv(a: np.ndarray, fps: float, window_s: float = 0.15) -> np.ndarray:
    """Smoothed first derivative (units per second); NaN where input is NaN."""
    win = max(5, int(round(fps * window_s)) | 1)
    out = np.full(len(a), np.nan)
    good = ~np.isnan(a)
    if good.sum() > win:
        out[good] = savgol_filter(a[good], win, 2, deriv=1, delta=1.0 / fps)
    return out


def _runs(mask: np.ndarray) -> list[tuple[int, int]]:
    """Contiguous True runs as (start, end_exclusive)."""
    m = np.concatenate([[False], np.asarray(mask, bool), [False]])
    d = np.diff(m.astype(int))
    return list(zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1)))


def _asym(left: float, right: float) -> float:
    if not (np.isfinite(left) and np.isfinite(right)) or left + right <= 0:
        return float("nan")
    return abs(left - right) / ((left + right) / 2) * 100


def _stance_light(index: float) -> str:
    if np.isnan(index):
        return "retake"
    if index <= STANCE_GREEN_MAX:
        return "green"
    if index <= STANCE_AMBER_MAX:
        return "amber"
    return "red"


# ---------------------------------------------------------------------------
# 1. Stance-time asymmetry
# ---------------------------------------------------------------------------
def analyse_stance(kps: dict, fps: float, mapping: dict = SUPERANIMAL_QUADRUPED) -> StanceResult:
    warnings: list = []
    cx, _, body_len = _body_and_scale(kps, fps, mapping, warnings)
    n = len(cx)
    body_speed = float(np.nanmedian(np.abs(_deriv(cx, fps)))) / body_len if n else float("nan")

    mean_s, counts, stances = {}, {}, {}
    if not np.isfinite(body_speed) or body_speed < MIN_WALK_SPEED_BL:
        warnings.append("Cat was not walking steadily - film it walking past the camera.")
        body_speed = float(body_speed) if np.isfinite(body_speed) else float("nan")
        empty = {p: float("nan") for p in PAWS}
        return StanceResult(empty, {p: 0 for p in PAWS}, {p: [] for p in PAWS},
                            float("nan"), float("nan"), float("nan"), None, "retake",
                            body_speed, warnings)

    thr = STANCE_SPEED_FRAC * body_speed
    stopped = 0
    for paw in PAWS:
        r = _role(kps, mapping, f"{paw}_paw", fps)
        if r is None:
            mean_s[paw], counts[paw], stances[paw] = float("nan"), 0, []
            warnings.append(f"{paw.replace('_', ' ')} paw not found.")
            continue
        speed = np.abs(_deriv(r[0], fps)) / body_len
        on_ground = np.where(np.isnan(speed), False, speed < thr)
        durs, spans = [], []
        for s, e in _runs(on_ground):
            if s == 0 or e == n:          # incomplete stance at clip edge
                continue
            # sub-frame start/end: interpolate where speed crosses the threshold
            a0, a1 = speed[s - 1], speed[s]
            b0, b1 = speed[e - 1], speed[e]
            start = s - 1 + ((a0 - thr) / (a0 - a1) if np.isfinite(a0) and a0 != a1 else 1.0)
            end = e - 1 + ((thr - b0) / (b1 - b0) if np.isfinite(b1) and b1 != b0 else 1.0)
            d = (end - start) / fps
            if d < MIN_STANCE_S:
                continue
            if d > MAX_STANCE_S:
                stopped += 1
                continue
            durs.append(d)
            spans.append((start / fps, end / fps))
        counts[paw] = len(durs)
        stances[paw] = spans
        mean_s[paw] = float(np.mean(durs)) if len(durs) >= MIN_STANCES_PER_PAW else float("nan")
    if stopped:
        warnings.append("The cat stopped during the clip; stops were ignored.")
    if any(counts[p] < MIN_STANCES_PER_PAW for p in PAWS):
        warnings.append(f"Fewer than {MIN_STANCES_PER_PAW} full steps seen for some paws - film a longer walk.")

    fa = _asym(mean_s["front_left"], mean_s["front_right"])
    ha = _asym(mean_s["hind_left"], mean_s["hind_right"])
    front_mean = np.nanmean([mean_s["front_left"], mean_s["front_right"]]) \
        if np.isfinite(mean_s["front_left"]) or np.isfinite(mean_s["front_right"]) else np.nan
    hind_mean = np.nanmean([mean_s["hind_left"], mean_s["hind_right"]]) \
        if np.isfinite(mean_s["hind_left"]) or np.isfinite(mean_s["hind_right"]) else np.nan
    fh = float(front_mean / hind_mean) if np.isfinite(front_mean) and np.isfinite(hind_mean) and hind_mean > 0 else float("nan")

    lights = [l for l in (_stance_light(fa), _stance_light(ha)) if l != "retake"]
    light = max(lights, key=ORDER.get) if lights else "retake"

    limb = None
    if light in ("amber", "red"):
        end = "front" if np.nan_to_num(fa) >= np.nan_to_num(ha) else "hind"
        left, right = mean_s[f"{end}_left"], mean_s[f"{end}_right"]
        limb = f"{end} {'left' if left > right else 'right'}"   # longer stance = sore side

    return StanceResult(mean_s, counts, stances, fa, ha, fh, limb, light, body_speed, warnings)


# ---------------------------------------------------------------------------
# 2. Jump check
# ---------------------------------------------------------------------------
def _jump_retake(reason, t, up, warnings, **kw) -> JumpResult:
    base = dict(jumped=False, takeoff_s=float("nan"), landing_s=float("nan"),
                hesitation_s=float("nan"), jump_height_bl=float("nan"), n_stages=0,
                two_stage=False, aborted_attempts=0, push_off_diff_s=float("nan"))
    base.update(kw)
    return JumpResult("retake", reason, flags=[], t=t, centre_up=up, warnings=warnings, **base)


def analyse_jump(kps: dict, fps: float, mapping: dict = SUPERANIMAL_QUADRUPED) -> JumpResult:
    """Side view of a cat jumping up onto a surface (sofa, bed, chair)."""
    warnings: list = []
    cx, cy, body_len = _body_and_scale(kps, fps, mapping, warnings)
    n = len(cy)
    t = np.arange(n) / fps
    up = -cy / body_len
    x = cx / body_len

    if n / fps < MIN_CLIP_S:
        return _jump_retake(f"Clip too short (under {MIN_CLIP_S:g} s). Film from the approach until the cat has settled on top.",
                            t, up, warnings)
    coverage = float(np.mean(~np.isnan(up)))
    if coverage < 0.6:
        return _jump_retake(f"Cat tracked in only {coverage:.0%} of frames. Film again with the whole cat in view.",
                            t, up, warnings)

    vy = _deriv(up, fps)
    vx = _deriv(x, fps)
    speed = np.hypot(vx, vy)

    # --- rises (fast upward movement), merged across short dips ---------------
    rising = np.where(np.isnan(vy), False, vy > RISE_SPEED_BL)
    runs = _runs(rising)
    merged: list[list[int]] = []
    for s, e in runs:
        if merged and (s - merged[-1][1]) / fps < PAUSE_MIN_S:
            merged[-1][1] = e
        else:
            merged.append([s, e])
    stages = [(s, e) for s, e in merged
              if np.nan_to_num(up[min(e, n - 1)] - up[s]) >= MIN_STAGE_RISE_BL]
    total_rise = sum(up[min(e, n - 1)] - up[s] for s, e in stages)
    jumped = bool(stages) and total_rise >= MIN_JUMP_RISE_BL

    takeoff = stages[0][0] if jumped else n
    pre = up[:takeoff]
    if np.sum(~np.isnan(pre)) < max(3, int(0.2 * fps)):
        pre = up
        if jumped:
            warnings.append("Take-off happened right at the start of the clip; start filming earlier.")
    standing = float(np.nanpercentile(pre, 90))
    lowered = np.where(np.isnan(up), False, up < standing - CROUCH_DROP_BL)

    # --- aborted attempts: crouches before take-off that end without a jump ---
    aborted, crouched_at_end = 0, False
    for s, e in _runs(lowered[:takeoff]):
        if (e - s) / fps < MIN_CROUCH_S:
            continue
        if jumped and (takeoff - e) / fps <= ABORT_GAP_S:
            continue                      # the crouch that led into the jump
        if not jumped and e == n:
            crouched_at_end = True        # still crouched when the clip ended
            continue
        aborted += 1

    if not jumped:
        if aborted == 0:
            msg = ("Still crouched when the clip ended - keep filming until the cat jumps."
                   if crouched_at_end else
                   "No jump found. Film the cat jumping onto a sofa or bed from the side.")
            return _jump_retake(msg, t, up, warnings)
        flags = ["Started to jump but did not jump" + (f" ({aborted} tries)" if aborted > 1 else "")]
        msg = (f"{flags[0]}. Cats that avoid jumping are often uncomfortable - "
               f"we recommend a vet check. {DISCLAIMER}")
        return JumpResult("red", msg, False, float("nan"), float("nan"), float("nan"),
                          float("nan"), 0, False, aborted, float("nan"), flags, t, up, warnings)

    # --- landing plateau --------------------------------------------------------
    last_end = stages[-1][1]
    settled = np.flatnonzero(np.where(np.isnan(vy[last_end:]), False,
                                      np.abs(vy[last_end:]) < SETTLE_SPEED_BL))
    if settled.size == 0:
        return _jump_retake("The clip ends before the cat has landed. Keep filming until it settles on top.",
                            t, up, warnings, jumped=True, takeoff_s=takeoff / fps)
    land = last_end + int(settled[0])
    plateau = up[land:land + max(2, int(0.3 * fps))]
    if (n - land) / fps < 0.2:
        warnings.append("Clip ends right after landing; height may be inaccurate.")
    height = float(np.nanmedian(plateau) - standing)

    # --- hesitation: still or crouched time just before take-off --------------
    i = takeoff - 1
    while i >= 0 and (lowered[i] or (np.isfinite(speed[i]) and speed[i] < STILL_SPEED_BL)):
        i -= 1
    hesitation = (takeoff - (i + 1)) / fps
    if i < 0:
        warnings.append("Cat was already waiting when filming started; hesitation may be longer.")

    # --- hind push-off symmetry -----------------------------------------------
    push_diff = float("nan")
    lifts = []
    for paw in ("hind_left", "hind_right"):
        r = _role(kps, mapping, f"{paw}_paw", fps)
        if r is None:
            lifts.append(None)
            continue
        p_up = -r[1] / body_len
        g0, g1 = max(0, takeoff - int(0.5 * fps)), max(1, takeoff - int(0.1 * fps))
        ground = np.nanmedian(p_up[g0:g1]) if np.any(~np.isnan(p_up[g0:g1])) else np.nan
        start = max(0, takeoff - int(0.3 * fps))
        above = np.flatnonzero(np.nan_to_num(p_up[start:] - ground, nan=-1) > LIFT_HEIGHT_BL)
        lifts.append((start + above[0]) / fps if above.size and np.isfinite(ground) else None)
    if None in lifts:
        warnings.append("Could not see both hind paws lift off; push-off symmetry not measured.")
    else:
        push_diff = abs(lifts[0] - lifts[1])

    # --- traffic light ------------------------------------------------------------
    flags, levels = [], []
    if hesitation > HESITATION_AMBER_MAX_S:
        flags.append(f"Hesitated before jumping ({hesitation:.1f} s)"); levels.append("red")
    elif hesitation > HESITATION_GREEN_MAX_S:
        flags.append(f"Hesitated before jumping ({hesitation:.1f} s)"); levels.append("amber")
    two_stage = len(stages) >= 2
    if two_stage:
        flags.append("Climbed in two stages" if len(stages) == 2 else f"Climbed in {len(stages)} stages")
        levels.append("amber")
    if aborted:
        flags.append(f"Started to jump but stopped first ({aborted}x)"); levels.append("amber")
    if np.isfinite(push_diff) and push_diff > PUSH_OFF_DIFF_MAX_S:
        flags.append("Hind legs pushed off unevenly"); levels.append("amber")

    if "red" in levels or len(levels) >= 2:
        light = "red"
    elif levels:
        light = "amber"
    else:
        light = "green"

    if light == "green":
        msg = "Smooth jump. Keep checking regularly."
    elif light == "amber":
        msg = (f"{'. '.join(flags)}. Film again in a few days; mention it to your vet "
               f"if it keeps happening. {DISCLAIMER}")
    else:
        msg = (f"{'. '.join(flags)}. Changes like these are common signs of joint "
               f"discomfort in cats - we recommend a vet check. {DISCLAIMER}")

    return JumpResult(light, msg, True, takeoff / fps, land / fps, float(hesitation), height,
                      len(stages), two_stage, aborted, push_diff, flags, t, up, warnings)


# ---------------------------------------------------------------------------
# 3. Combined cat check
# ---------------------------------------------------------------------------
def analyse_cat(kps: dict, fps: float, mode: str = "walk",
                mapping: dict = SUPERANIMAL_QUADRUPED) -> CatResult:
    if mode == "jump":
        j = analyse_jump(kps, fps, mapping)
        return CatResult("jump", j.light, None, j.message, jump=j, warnings=list(j.warnings))
    if mode != "walk":
        raise ValueError('mode must be "walk" or "jump"')

    warnings: list = []
    try:
        g = analyse_gait(kps, fps, mapping)
        warnings += [w.replace("Dog", "Cat") for w in g.warnings]
    except ValueError as e:
        g = None
        warnings.append(f"Head/pelvis check skipped: {e}")
    s = analyse_stance(kps, fps, mapping)
    warnings += s.warnings

    parts = []                                   # (light, limb, source)
    if g is not None and g.light != "retake":
        parts.append((g.light, g.suspected_limb, "gait"))
    if s.light != "retake":
        parts.append((s.light, s.suspected_limb, "stance"))

    if not parts:
        return CatResult("walk", "retake", None,
                         "Could not read the walk reliably. Film your cat walking past the "
                         "camera side-on for at least 3-4 seconds (a treat at the end of a "
                         "hallway helps).", g, s, None, warnings)

    # worst light wins; on a tie prefer stance (more sensitive for cats)
    light, limb, source = max(parts, key=lambda p: (ORDER[p[0]], p[2] == "stance"))
    where = ""
    if limb:
        why = " - that paw stays on the ground longer" if source == "stance" else ""
        where = f" ({limb}{why})"
    messages = {
        "green": "Walking looks even. Cats hide discomfort well, so also try the jump check "
                 "and keep checking regularly.",
        "amber": f"Slight movement changes{where}. Re-check in a few days; mention it to your "
                 f"vet if it persists. {DISCLAIMER}",
        "red": f"Clear movement changes{where}. We recommend a vet check. {DISCLAIMER}",
    }
    return CatResult("walk", light, limb, messages[light], g, s, None, warnings)


In [ ]:
%%writefile synthetic_cat.py
"""Synthetic cat keypoints for testing the cat checks (SuperAnimal-Quadruped names).

make_walk : lateral-sequence walk (footfalls LH, LF, RH, RF, a quarter stride apart)
            with an optional limb that stays on the ground longer (Stadig et al. 2016).
make_jump : side view of a cat jumping onto a surface: "smooth", "hesitant"
            (long crouch), "two_stage" (rise, pause, second rise) or "aborted"
            (crouch, then walks away without jumping).
"""
import numpy as np

GROUND_Y = 600.0
FOOTFALL = {"hind_left": 0.0, "front_left": 0.25, "hind_right": 0.5, "front_right": 0.75}
DLC_NAME = {"front_left": "front_left_paw", "front_right": "front_right_paw",
            "hind_left": "back_left_paw", "hind_right": "back_right_paw"}


def _add_body(out, add, cx, back_y, head_up, pelvis_up, body_px, direction):
    front = 0.45 * body_px * direction
    back = -0.45 * body_px * direction
    head_y = back_y - 0.25 * body_px - head_up
    for nm, dx in [("left_earbase", 0), ("right_earbase", 3),
                   ("left_eye", 15 * direction), ("right_eye", 17 * direction)]:
        add(nm, cx + front + 0.15 * body_px * direction + dx, head_y + (8 if "eye" in nm else 0))
    add("neck_base", cx + front, back_y - 0.3 * head_up)
    add("tail_base", cx + back, back_y - pelvis_up)
    return front, back


def make_walk(long_stance: str | None = None, severity: float = 0.5, seconds: float = 8.0,
              fps: float = 30.0, stride: float = 0.9, duty: float = 0.6,
              body_px: float = 200.0, speed_bl: float = 1.0, noise_px: float = 1.0,
              drift_px: float = 8.0, direction: int = 1, seed: int = 0):
    """Return ({bodypart: (x, y, p)}, fps).

    long_stance: "front_left" / "front_right" / "hind_left" / "hind_right" / None.
    That paw's stance lasts (1 + 0.4 * severity) times longer (footfall timing kept).
    Head/pelvis bob is kept symmetric on purpose: cats often hide a limp.
    """
    rng = np.random.default_rng(seed)
    n = int(seconds * fps)
    t = np.arange(n) / fps
    v = speed_bl * body_px * direction
    cx = (100 if direction > 0 else 1800) + v * t
    drift = drift_px * np.sin(2 * np.pi * t / 3.0)

    amp_head, amp_pelvis = 0.03 * body_px, 0.02 * body_px
    half = stride / 2
    pelvis_up = -amp_pelvis * np.cos(2 * np.pi * (t - duty / 2 * stride) / half)
    head_up = -amp_head * np.cos(2 * np.pi * (t - (0.25 + duty / 2) * stride) / half)

    out = {}

    def add(name, x, y):
        out[name] = (x + rng.normal(0, noise_px, n),
                     y + drift + rng.normal(0, noise_px, n),
                     np.full(n, 0.9))

    back_y = GROUND_Y - 0.6 * body_px
    front, back = _add_body(out, add, cx, back_y, head_up, pelvis_up, body_px, direction)

    for paw, phi in FOOTFALL.items():
        d = min(0.9, duty * (1 + 0.4 * severity)) if paw == long_stance else duty
        # integrate paw speed at 20x the frame rate so stance edges are not frame-quantised
        sub = 20
        ts = np.arange(n * sub) / (fps * sub)
        stance = ((ts / stride - phi) % 1.0) < d
        xs = np.cumsum(np.where(stance, 0.0, v / (1 - d))) / (fps * sub)
        x = xs[::sub]
        x = x - np.mean(x - v * t)                    # keep the paw under the body
        off = (front if paw.startswith("front") else back) + (5 if "right" in paw else 0)
        add(DLC_NAME[paw], cx[0] + off + x, np.full(n, GROUND_Y))
    return out, fps


def _ease(u):
    u = np.clip(u, 0, 1)
    return 3 * u ** 2 - 2 * u ** 3


JUMP_KINDS = ("smooth", "hesitant", "two_stage", "aborted")


def make_jump(kind: str = "smooth", seconds: float | None = None, fps: float = 30.0,
              body_px: float = 200.0, height_px: float | None = None,
              push_delay_s: float = 0.0, noise_px: float = 1.0, seed: int = 0):
    """Return ({bodypart: (x, y, p)}, fps) for a cat jumping onto a surface.

    seconds: optionally cut the clip to this length (for "too short" tests).
    push_delay_s: right hind paw leaves the ground this much later than the left.
    """
    if kind not in JUMP_KINDS:
        raise ValueError(f"kind must be one of {JUMP_KINDS}")
    rng = np.random.default_rng(seed)
    H = 1.5 * body_px if height_px is None else height_px
    crouch = 0.15 * body_px
    walk_v = 1.0 * body_px

    # segments: (mode, duration s, dx px, dy px up)
    seg = [("walk", 1.5, walk_v * 1.5, 0.0), ("ease", 0.25, 0.0, -crouch)]
    if kind == "smooth":
        seg += [("hold", 0.5, 0, 0), ("ease", 0.3, 0.8 * body_px, H + crouch), ("hold", 1.5, 0, 0)]
    elif kind == "hesitant":
        seg = [("walk", 1.0, walk_v, 0.0), ("ease", 0.25, 0.0, -crouch),
               ("hold", 4.0, 0, 0), ("ease", 0.3, 0.8 * body_px, H + crouch), ("hold", 1.5, 0, 0)]
    elif kind == "two_stage":
        first = 0.5 * H
        seg += [("hold", 0.5, 0, 0), ("ease", 0.3, 0.5 * body_px, first + crouch),
                ("hold", 0.6, 0, 0), ("ease", 0.5, 0.3 * body_px, H - first), ("hold", 1.5, 0, 0)]
    elif kind == "aborted":
        seg += [("hold", 1.5, 0, 0), ("ease", 0.4, 0.0, crouch), ("walk", 2.0, -walk_v * 2.0, 0.0)]

    total = sum(s[1] for s in seg)
    n = int(round(total * fps))
    t = np.arange(n) / fps
    cx = np.zeros(n)
    up = np.zeros(n)
    x0, y0, t0 = 100.0, 0.0, 0.0
    for mode, dur, dx, dy in seg:
        m = (t >= t0) & (t < t0 + dur)
        u = (t[m] - t0) / dur
        if mode == "walk":
            cx[m] = x0 + dx * u
            up[m] = y0 + 0.015 * body_px * np.sin(2 * np.pi * 2.2 * (t[m] - t0))
        elif mode == "ease":
            cx[m] = x0 + dx * _ease(u)
            up[m] = y0 + dy * _ease(u)
        else:
            cx[m] = x0
            up[m] = y0
        x0, y0, t0 = x0 + dx, y0 + dy, t0 + dur

    stand_h = 0.6 * body_px                    # centre height above the paws when standing
    centre_y = GROUND_Y - stand_h - up         # image y (down)

    out = {}

    def add(name, x, y):
        out[name] = (x + rng.normal(0, noise_px, n), y + rng.normal(0, noise_px, n),
                     np.full(n, 0.9))

    # body kept level for simplicity (no head/pelvis bob during the jump)
    front, back = _add_body(out, add, cx, centre_y, np.zeros(n), np.zeros(n), body_px, 1)

    # paws hang a leg-length below the centre but cannot go below the floor;
    # once on top, the surface is at centre + leg length so the same rule holds.
    def paw_y(delay):
        shift = int(round(delay * fps))
        cy = np.concatenate([np.full(shift, centre_y[0]), centre_y[:n - shift]]) if shift else centre_y
        return np.minimum(GROUND_Y, cy + stand_h)

    add("front_left_paw", cx + front, paw_y(0.0))
    add("front_right_paw", cx + front + 5, paw_y(0.0))
    add("back_left_paw", cx + back, paw_y(0.0))
    add("back_right_paw", cx + back + 5, paw_y(push_delay_s))

    if seconds is not None:
        k = int(round(seconds * fps))
        out = {nm: (a[:k], b[:k], c[:k]) for nm, (a, b, c) in out.items()}
    return out, fps


In [ ]:
import importlib, cat_analysis, synthetic_cat
for m in (gait_analysis, cat_analysis, synthetic_cat): importlib.reload(m)
from cat_analysis import analyse_cat
from synthetic_cat import make_walk, make_jump

# self-test on simulated cats
for label, (k, f), mode in [("sound walk", make_walk(), "walk"),
                            ("hind-left long stance", make_walk(long_stance="hind_left", severity=0.6), "walk"),
                            ("smooth jump", make_jump("smooth"), "jump"),
                            ("hesitant jump", make_jump("hesitant"), "jump"),
                            ("two-stage climb", make_jump("two_stage"), "jump")]:
    r = analyse_cat(k, f, mode)
    print(f"simulated {label:22s} -> {r.light:6s} {r.suspected_limb or ''}  | {r.message[:70]}")

In [ ]:
# Run on the video analysed in section 4
cat_mode = "walk"  # @param ["walk", "jump"]
import matplotlib.pyplot as plt
cat = analyse_cat(kps, fps, cat_mode)
icon = {"green": "🟢", "amber": "🟠", "red": "🔴", "retake": "⚪"}[cat.light]
print(f"{icon} {cat.light.upper()}  {cat.suspected_limb or ''}\n{cat.message}")
if cat.stance:
    s = cat.stance
    print("mean stance (s):", {k: round(v, 2) for k, v in s.mean_stance_s.items()},
          f"| front asym {s.front_asymmetry:.0f}%  hind asym {s.hind_asymmetry:.0f}%")
if cat.jump:
    j = cat.jump
    print(f"hesitation {j.hesitation_s:.1f}s | height {j.jump_height_bl:.2f} body lengths | stages {j.n_stages} | "
          f"aborted {j.aborted_attempts} | push-off diff {j.push_off_diff_s:.2f}s | flags: {j.flags}")
    plt.figure(figsize=(10, 3)); plt.plot(j.t, j.centre_up); plt.title("Body centre height"); plt.xlabel("time (s)")
    for x in (j.takeoff_s, j.landing_s):
        if x == x: plt.axvline(x, ls="--", c="grey")
for w in cat.warnings:
    print("⚠", w)

## 6b. Demo batch (pitch videos)
Put real owner videos (e.g. from Reddit) in `/content/demo_in/` named `cat1.mp4`, `dog1.mp4`, …
(drag them into the Files panel). `demo=True` draws a hand-written red "Ask your vet" banner with a small DEMO tag;
the table still shows what the model itself measured. Optional `trims = {"dog1": (start_s, duration_s)}`.

In [ ]:
%%writefile run_demo.py
"""Batch demo: every video in a folder -> skeleton + analysis + overlay video.

File name decides the species: names starting with "cat" use the cat walk check,
everything else the dog gait check. Run on Colab (GPU) after DeepLabCut is installed:

    from run_demo import run_folder
    table = run_folder("/content/demo_in", "/content/demo_out", demo=True)

demo=True  -> banner is a hand-written DemoLabel (red "Ask your vet") for the pitch,
              the model's real numbers are still returned in the table.
demo=False -> banner shows the model's own verdict.
"""
from __future__ import annotations

import glob
import os
import subprocess

import cv2

from cat_analysis import analyse_cat
from gait_analysis import analyse_gait, keypoints_from_dlc_h5
from visualize import DemoLabel, render_video

DEMO_LABELS = {
    "dog": DemoLabel("red", "Ask your vet", "Uneven gait - possible limp"),
    "cat": DemoLabel("red", "Ask your vet", "Movement change - possible limp"),
}


def _normalise(src: str, dst: str, start: float = 0, duration: float = 0):
    trim = (["-ss", str(start)] if start else []) + (["-t", str(duration)] if duration else [])
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", *trim, "-i", src,
                    "-vf", "scale='min(1280,iw)':-2", "-r", "30", "-an",
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", dst], check=True)


def run_one(src: str, out_dir: str, demo: bool = True, trims: dict | None = None):
    from deeplabcut.modelzoo.video_inference import video_inference_superanimal

    name = os.path.splitext(os.path.basename(src))[0].replace(" ", "_")
    species = "cat" if name.lower().startswith("cat") else "dog"
    work = os.path.join(out_dir, "work")
    os.makedirs(work, exist_ok=True)
    vid = os.path.join(work, name + ".mp4")
    start, dur = (trims or {}).get(name, (0, 0))
    _normalise(src, vid, start, dur)

    video_inference_superanimal(videos=[vid], superanimal_name="superanimal_quadruped",
                                model_name="hrnet_w32", detector_name="fasterrcnn_resnet50_fpn_v2",
                                max_individuals=1, dest_folder=work, create_labeled_video=False)
    h5 = sorted(glob.glob(os.path.join(work, name + "*.h5")), key=os.path.getmtime)[-1]
    fps = cv2.VideoCapture(vid).get(cv2.CAP_PROP_FPS)
    kps = keypoints_from_dlc_h5(h5)

    if species == "cat":
        res = analyse_cat(kps, fps, "walk")
        head = res.gait.head.asymmetry_index if res.gait else float("nan")
        pelvis = res.gait.pelvis.asymmetry_index if res.gait else float("nan")
    else:
        res = analyse_gait(kps, fps)
        head, pelvis = res.head.asymmetry_index, res.pelvis.asymmetry_index

    raw = os.path.join(work, name + "_overlay_raw.mp4")
    final = os.path.join(out_dir, name + ("_demo.mp4" if demo else "_model.mp4"))
    render_video(vid, kps, res, raw, demo_label=DEMO_LABELS[species] if demo else None)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw, "-c:v", "libx264",
                    "-pix_fmt", "yuv420p", final], check=True)
    return dict(video=name, species=species, model_light=res.light, model_limb=res.suspected_limb,
                head_asym=round(head, 1), pelvis_asym=round(pelvis, 1),
                message=res.message, warnings="; ".join(res.warnings), output=final)


def run_folder(in_dir: str, out_dir: str, demo: bool = True, trims: dict | None = None):
    import pandas as pd

    os.makedirs(out_dir, exist_ok=True)
    rows = []
    for src in sorted(glob.glob(os.path.join(in_dir, "*.mp4")) + glob.glob(os.path.join(in_dir, "*.mov"))):
        try:
            rows.append(run_one(src, out_dir, demo, trims))
        except Exception as e:  # keep going with the other videos
            rows.append(dict(video=os.path.basename(src), error=repr(e)))
    return pd.DataFrame(rows)


In [ ]:
import importlib, visualize, run_demo
for m in (gait_analysis, cat_analysis, visualize, run_demo): importlib.reload(m)
os.makedirs("/content/demo_in", exist_ok=True)
trims = {}  # e.g. {"dog1": (2, 8)}
table = run_demo.run_folder("/content/demo_in", "/content/demo_out", demo=True, trims=trims)
table

In [ ]:
# Download all demo videos as one zip
import shutil
shutil.make_archive("/content/demo_videos", "zip", "/content/demo_out", ".")
files.download("/content/demo_videos.zip")

## 7. Calibrate thresholds on several videos
This answers *"how do we know what is a good or bad gait?"*: the pose dataset has no healthy/lame labels, so **we label our own videos**.

1. Put videos in Google Drive, named with their label: `healthy_bella.mp4`, `lame_frontleft_max.mp4`, …
2. Run the cells below → a table of asymmetry per video.
3. Set `GREEN_MAX` just above the healthy dogs' values and check the lame dogs land above it.

Best label source: a vet watching the same clips and saying lame / not lame.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
video_dir = "/content/drive/MyDrive/gait-videos"  # @param {type:"string"}

In [ ]:
import pandas as pd

rows = []
for src in sorted(glob.glob(f"{video_dir}/*.mp4") + glob.glob(f"{video_dir}/*.mov")):
    name = os.path.splitext(os.path.basename(src))[0].replace(" ", "_")
    dst = f"/content/work/{name}.mp4"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-vf", "scale='min(1280,iw)':-2",
                    "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", dst], check=True)
    video_inference_superanimal(videos=[dst], superanimal_name="superanimal_quadruped", model_name="hrnet_w32",
                                detector_name="fasterrcnn_resnet50_fpn_v2", max_individuals=1,
                                dest_folder=out_dir, create_labeled_video=False)
    h5 = sorted(glob.glob(f"{out_dir}/{name}*.h5"), key=os.path.getmtime)[-1]
    r = analyse_gait(keypoints_from_dlc_h5(h5), cv2.VideoCapture(dst).get(cv2.CAP_PROP_FPS))
    rows.append(dict(video=name, label=name.split("_")[0], light=r.light, limb=r.suspected_limb,
                     head_asym=round(r.head.asymmetry_index, 1), pelvis_asym=round(r.pelvis.asymmetry_index, 1),
                     steps=min(r.head.n_steps, r.pelvis.n_steps), warnings="; ".join(r.warnings)))
table = pd.DataFrame(rows)
table.to_csv("/content/work/calibration.csv", index=False)
table

In [ ]:
# Current thresholds (edit and re-run section 5/6 to try new ones)
import gait_analysis
print("GREEN_MAX =", gait_analysis.GREEN_MAX, " AMBER_MAX =", gait_analysis.AMBER_MAX)
if len(table):
    print(table.groupby("label")[["head_asym", "pelvis_asym"]].describe().round(1))